In [ ]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:98% !important;}
div.cell.code_cell.rendered{width:98%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render li, div.text_cell_render p{width:95% !important;font-size:20pt;padding:5px; line-height:30px;}
table.dataframe{font-size:22px;}
table td, th{font-size:16px;}
table{ margin-left:0 !important;   /* 왼쪽 여백 0 */}
</style>
"""))

# 1. knowledge base 구성을 위한 데이터 생성

In [2]:
%%time
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = Docx2txtLoader("tax/소득세법_with_markdown.docx")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=20
)

documents = loader.load_and_split(text_splitter=text_splitter)
len(documents)

CPU times: total: 2.98 s
Wall time: 2.99 s


180

In [3]:
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings
load_dotenv(override=True) # 커널을 시작하지 않고 .env덮어쓰기
embedding = UpstageEmbeddings(
    model = 'embedding-query'
)

In [4]:
len(embedding.embed_query("소득세법"))

4096

In [5]:
from pinecone import Pinecone
pc = Pinecone()
[index.name for index in pc.list_indexes()]

['tax-index-openai', 'tax-index', 'tax-index-table']

In [6]:
# pinecone의 index 생성하기
import time
from pinecone import Pinecone, ServerlessSpec

# pinecone 객체 생성
pc = Pinecone() # api_key=os.getenv('PINECONE_API_KEY')

# 인덱스 설정
index_name = 'tax-index-markdown'
dimension = len(embedding.embed_query("차원 확인용"))

# 인덱스 생성 (없을 경우에만 생성)
existing_indexes = [index.name for index in pc.list_indexes()]

if index_name not in existing_indexes:
    pc.create_index(
        name=index_name,
        dimension=dimension,
        metric="cosine",
        spec=ServerlessSpec(
            cloud='aws',
            region='us-east-1'
        )
    )
    while not pc.describe_index(index_name).status['ready']:
        time.sleep(1)
    print(f'인덱스 생성 : {index_name}')
else:
    print(f'중복된 인덱스 : {index_name}')

인덱스 생성 : tax-index-markdown


In [7]:
# 인덱스 정보 확인
print([index.name for index in pc.list_indexes()])
print(pc.describe_index(index_name).status['ready'])
index = pc.Index(index_name)
print(index.describe_index_stats())

['tax-index-openai', 'tax-index-markdown', 'tax-index', 'tax-index-table']
True


d:\miniconda\envs\llm\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'dimension': 4096,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'total_vector_count': 0,
 'vector_type': 'dense'}


In [8]:
%%time
# 문서를 임베딩해서 클라우드 vector DB(index)에 저장
from pinecone import Pinecone
pc = Pinecone()
from langchain_pinecone import PineconeVectorStore
database = PineconeVectorStore.from_documents(
    documents=documents,
    embedding=embedding,
    index_name=index_name
)

CPU times: total: 4.67 s
Wall time: 17.4 s


In [ ]:
# 업로드한 벡터DB를 load할 때
from pinecone import Pinecone
pc = Pinecone()
from langchain_pinecone import PineconeVectorStore
index_name = 'tax-index-table'
database = PineconeVectorStore(
    embedding=embedding,
    index_name=index_name
)

# 2. 답변 생성 전 retrieval 확인

In [ ]:
query = "연봉이 5천만원인 직장인의 소득세는 얼마인가요?"
retrieved_docs = database.similarity_search(query, k=2) # 기본k값은 4

In [10]:
retriever = database.as_retriever( 
    search_type='similarity',
    search_kwargs={'k':2}
)
retrieved_docs = retriever.invoke("연봉이 5천만원인 직장인의 소득세는 얼마인가요?")

In [11]:
retrieved_doc = "\n--\n".join([doc.page_content for doc in retrieved_docs])

In [12]:
'55조' in retrieved_doc

True

# 3. 답변 생성

In [ ]:
from openai import OpenAI
client = OpenAI()
query = "연봉이 5천만원인 직장인의 소득세는 얼마인가요?"
response = client.chat.completions.create(
    model = "gpt-4o-mini",
    messages=[
        {'role':'system', 'content':'당신은 최고의 한국 소득세법 전문가입니다'},
        {
            'role':'user',
            'content':f"""- [context]를 참고해서 사용자의 질문에 답변해 주세요.
            - [context] : {retrieved_doc}
            - 질문 : {query}"""
        }
    ],
    temperature=0.2
)

In [ ]:
result = response.choices[0].message.content

In [13]:
from langchain_upstage import ChatUpstage

llm = ChatUpstage(
    model = 'solar-pro4',
    reasoning_effort='low'
)

In [ ]:
prompt = f"""[identity]
- 당신은 최고의 한국 소득세법 전문가입니다.
- 다음의 [context]를 참고하여 사용자의 질문에 답변해 주세요.
{retrieved_doc}
- 질문 : {query}"""

In [15]:
ai_message = llm.invoke(prompt)
print(ai_message.content)

Overall, I will provide the calculated tax, noting that this is a simplified estimation based on a single-income worker with no additional deductions, and that actual taxes may differ. I'll also cite the relevant articles from the context.

Let's structure the answer:
- Explain the process: calculate tax base after earned income deduction and personal exemption, then apply progressive tax rates (Article 55), then apply earned income tax credit (Article 59).
- Show the computation for 50M salary.
- Give final tax amount.
- Note that actual tax might vary based on other deductions, dependents, and other tax credits.

We can mention that the context provides the tax rates and the earned income tax credit limits, but to calculate precisely, we use the law's standard deduction amounts from general knowledge (since the context didn't include them, but we are allowed to supplement). I'll note that.

Now, answer in Korean. The question is in Korean. So answer in Korean.

I'll start: "안녕하세요. 소득

# LangChain

In [16]:
# from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_upstage import ChatUpstage, UpstageEmbeddings
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv

# 임베딩 & LLM 초기화
load_dotenv()
# embedding = OpenAIEmbeddings(model = "text-embedding-3-large")
# llm = ChatOpenAI(model='gpt-4.1-mini')
embedding = UpstageEmbeddings(model='embedding-query')
llm = ChatUpstage(model='solar-pro4')

# vector DB load
# pc = Pinecone()
index_name = 'tax-index-markdown'
vectorStore = PineconeVectorStore(
    embedding=embedding,
    index_name=index_name
)

# Retriever 생성 - search_type='similarity' : 가장 유사한 top-k개 추출
retriever = vectorStore.as_retriever(search_type='similarity', 
                                     search_kwargs={'k':2}) # 유사도가 높은 top-2추출

# 프롬프트 템플릿
template = f"""당신은 최고의 한국 소득세법 전문가입니다. 다음 문맥을 참고하여 질문에 답하세요. 답을 모르면 모른다고 
답하세요. 최대 3문장으로 간결하게 답변하세요
질문 : {{query}}
문맥 : {{context}}
답변 :"""
prompt = ChatPromptTemplate.from_template(template)
# 검색된 문서를 텍스트로 변환하는 함수
def format_documents(retrieved_docs):
    return "\n--\n".join([doc.page_content.replace('\n\n', ' ') for doc in retrieved_docs])

In [17]:
# RAG 렝체인 구성 (LCEL방식)
rag_chain = (
    {
        "context":retriever | format_documents,
        "query":RunnablePassthrough() # 질문만 그대로 전달
    }
    | prompt
    | llm
    | StrOutputParser()
)
result = rag_chain.invoke('연봉 5천만원인 직장인의 소득세는 얼마인가요?')
print(result)

제공된 문맥만으로는 정확한 소득세를 계산할 수 없으므로 답변을 드릴 수 없습니다. 문맥에는 세율과 세액공제 규정만 포함되어 있을 뿐, 근로소득공제, 인적공제 등 과세표준 산출에 필요한 공제 정보가 누락되어 있습니다. 따라서 연봉 5천만원이라는 총급여만으로는 최종 소득세액을 특정할 수 없으며, 추가 정보가 필요합니다.


# 4. 키워드 사전 활용

In [18]:
query = '연봉 5천만원인 직장인의 소득세는 얼마인가요?'
# query = '종합소득이 5,000만원인 거주자의 소득세는 얼마인가요?'

In [19]:
import numpy as np
def cosine_similarity(vec1, vec2):
    '두 벡터의 코사인 유사도를 반환'
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    if norm_vec1==0 or norm_vec2==0:
        return 0
    return dot_product / (norm_vec1*norm_vec2)
vec1 = embedding.embed_query("직장인")
vec2 = embedding.embed_query("거주자")
cosine_similarity(vec1, vec2)

np.float64(0.5402861885922909)

In [20]:
vec1 = embedding.embed_query("연봉")
vec2 = embedding.embed_query("종합소득")
cosine_similarity(vec1, vec2)

np.float64(0.566316344244509)

In [21]:
vec1 = embedding.embed_query("5천")
vec2 = embedding.embed_query("5,000")
cosine_similarity(vec1, vec2)

np.float64(0.8728291248383045)

In [22]:
# 키워드 사전 : 일반사용자 단어 -> 소득세법 용어
from langchain_core.prompts import ChatPromptTemplate
dictionary = ["연봉->종합소득", "사람을 나타내는 표현->거주자", "5천->5,000"]
prompt = ChatPromptTemplate.from_template(f"""사용자의 질문을 보고, 우리의 사전을 참고해서 사용자 질문을
변경해 주세요. 만약 변경할 필요가 없을 경우, 사용자의 질문을 변경하지 않아도 됩니다. 오직 질문만 리턴해 주세요
- 사전 : {dictionary}
- 질문 : {{question}}""")

In [23]:
strOutputParser = StrOutputParser()
strOutputParser.invoke(llm.invoke(prompt.invoke({"question":"소득이 높은 남자가 있습니다"})))

'종합소득이 높은 거주자가 있습니다'

In [24]:
# langchain
dictionary_chain = prompt | llm | StrOutputParser()
#dictionary_chain.invoke({"question":"연봉이 5천만원인 직장인은?"})
dictionary_chain.invoke("연봉이 5천만원인 직장인은?")

'종합소득이 5,000만 원인 거주자는?'

In [25]:
new_chain = dictionary_chain | rag_chain

In [26]:
new_chain.invoke("연봉이 5천만원인 직장인의 소득세는 얼마예요?")
# new_chain.invoke({"question":"연봉이 5천만원인 직장인의 소득세는 얼마예요?"})

"제공된 문맥에 따르면 종합소득과세표준이 5,000만원인 경우 세율은 '84만원 + (1,400만원을 초과하는 금액의 15퍼센트)'가 적용됩니다. 따라서 산출세액은 84만원 + (3,600만원 × 15%) = 624만원이나, 정확한 소득세는 문맥에 없는 소득공제 및 세액공제를 반영한 과세표준 산정 후 결정됩니다."

```
입력 {"question":"연봉이 5천만원인 직장인의 소득세는 얼마예요?"}
       ↓
dictionary_chain에 의해 질문 수정(documents용어 수정)
       ↓
  "개선된 질문"
       ↓
  retriever
       ↓ 
format_documents
       ↓ 
rag_chain의 prompt + llm
      ↓ 
strOutputParser()
```